In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
sensor_bronze_df_path = "iot_krish.bronze.sensor_readings"
devices_bronze_df_path = "iot_krish.bronze.devices"
locations_bronze_df_path = "iot_krish.bronze.locations"

sensor_bronze = spark.table(sensor_bronze_df_path)
devices_bronze = spark.table(devices_bronze_df_path)
locations_bronze = spark.table(locations_bronze_df_path)

In [0]:
sensor_clean_df = (
    sensor_bronze
    .withColumn("reading_ts", to_timestamp("reading_ts"))
    .withColumn("reading_date", to_date("reading_ts"))
)

In [0]:
sensor_clean_df = sensor_clean_df.withColumn(
    "is_anomaly",
    when((col("temperature_c") > 90) | (col("vibration_hz") > 3.0), True).otherwise(False)
)

In [0]:
from pyspark.sql.window import Window

rolling_window = Window.partitionBy("device_id").orderBy("reading_ts").rowsBetween(-2, 0)

sensor_clean_df = sensor_clean_df.withColumn(
    "rolling_avg_temp",
    avg("temperature_c").over(rolling_window)
)

In [0]:
sensor_clean_join_df = sensor_clean_df \
    .join(devices_bronze, "device_id", "left") \
    .join(locations_bronze, "location_id", "left") \
    .select(
        sensor_clean_df["*"],
        devices_bronze["device_type"],
        devices_bronze["maintenance_due"],
        locations_bronze["zone_name"],
        locations_bronze["floor"],
        locations_bronze["supervisor"]
    )

In [0]:
(sensor_clean_join_df.write
 .format("delta")
 .mode("overwrite")
 .partitionBy("location_id")
 .option("overwriteSchema", "true")
 .saveAsTable("iot_krish.silver.sensor_readings"))

In [0]:
silver_devices = (
    devices_bronze
    .withColumn("effective_start_date", current_timestamp())
    .withColumn("effective_end_date", lit(None).cast("timestamp"))
    .withColumn("is_current", lit(True))
)

In [0]:
(silver_devices.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.silver.devices"))

In [0]:
devices_updates_path = "dbfs:/FileStore/iot_krish/raw/devices_update.json"

In [0]:
devices_updates_df = (
    spark.read
    .json(devices_updates_path)
)
display(devices_updates_df)

In [0]:
devices_updates_df.createOrReplaceTempView("devices_updates")

In [0]:
%sql
SELECT * FROM devices_updates;

In [0]:
%sql
MERGE INTO iot_krish.silver.devices AS target
USING devices_updates AS source
ON target.device_id = source.device_id
AND target.is_current = true

WHEN MATCHED AND target.maintenance_due <> source.maintenance_due THEN
  UPDATE SET
    target.is_current = false,
    target.effective_end_date = current_timestamp()

In [0]:
%sql
INSERT INTO iot_krish.silver.devices
SELECT
    s.device_id,
    s.device_type,
    s.location_id,
    s.installation_date,
    s.maintenance_due,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM devices_updates s
LEFT JOIN iot_krish.silver.devices t
    ON s.device_id = t.device_id
    AND t.is_current = true
    AND s.maintenance_due = t.maintenance_due
WHERE t.device_id IS NULL

In [0]:
%sql
SELECT *
FROM iot_krish.silver.devices
ORDER BY device_id, effective_start_date

In [0]:
%sql
ALTER TABLE iot_krish.silver.sensor_readings
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

In [0]:
%sql
DESCRIBE HISTORY iot_krish.silver.sensor_readings

In [0]:
%sql
OPTIMIZE iot_krish.silver.sensor_readings
ZORDER BY (device_id, reading_ts);

In [0]:
%sql
UPDATE iot_krish.silver.sensor_readings
SET status = 'critical'
WHERE temperature_c > 95

In [0]:
%sql
SELECT *
FROM table_changes('iot_krish.silver.sensor_readings', 8)